In [0]:
# this wil generate batches of json files in the folder 'incremental' of the volume. The goal here is to simulate a real incremental ingestion for new orders

In [0]:
dbutils.widgets.text("n_new_orders", "50", "New orders")
dbutils.widgets.text("n_status_updates", "30", "Status updates")
dbutils.widgets.dropdown("inject_bad_records", "true", ["true", "false"], "Inject bad records")

N_NEW_ORDERS = int(dbutils.widgets.get("n_new_orders"))
N_STATUS_UPDATES = int(dbutils.widgets.get("n_status_updates"))
INJECT_BAD_RECORDS = dbutils.widgets.get("inject_bad_records") == "true"

In [0]:
import random
import uuid
from datetime import datetime, timedelta
from decimal import Decimal

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

SILVER = "olist_lakehouse.silver"
INCREMENTAL_PATH = "/Volumes/olist_lakehouse/landing/olist/incremental"
BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
NOW = datetime.now()

def fmt(dt):
    """formats the dates accordingly to the one we have in the olist csvs"""
    return dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None

# columns of each table, same as in the original CSVs
COLUMNS = {
    "orders": [
        "order_id", "customer_id", "order_status", "order_purchase_timestamp",
        "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date",
    ],
    "order_items": [
        "order_id", "order_item_id", "product_id", "seller_id",
        "shipping_limit_date", "price", "freight_value",
    ],
    "order_payments": [
        "order_id", "payment_sequential", "payment_type",
        "payment_installments", "payment_value",
    ],
    "customers": [
        "customer_id", "customer_unique_id", "customer_zip_code_prefix",
        "customer_city", "customer_state",
    ],
}

In [0]:
customers_pool = (
    spark.table(f"{SILVER}.customers")
    .select("customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state")
    .orderBy(F.rand())
    .limit(1000)
    .collect()
)

items_pool = (
    spark.table(f"{SILVER}.order_items")
    .select("product_id", "seller_id", "price", "freight_value")
    .orderBy(F.rand())
    .limit(1000)
    .collect()
)

print(f"{len(customers_pool)} customers and {len(items_pool)} combinations product/vendor in this sample")

In [0]:
new_orders, new_items, new_payments, new_customers = [], [], [], []

for _ in range(N_NEW_ORDERS):
    order_id = uuid.uuid4().hex
    customer_id = uuid.uuid4().hex 
    customer = random.choice(customers_pool)
    purchase_ts = NOW - timedelta(minutes=random.randint(0, 60))

    new_customers.append({
        "customer_id": customer_id,
        "customer_unique_id": customer.customer_unique_id,
        "customer_zip_code_prefix": customer.customer_zip_code_prefix,
        "customer_city": customer.customer_city,
        "customer_state": customer.customer_state,
    })

    new_orders.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "order_status": "created",
        "order_purchase_timestamp": fmt(purchase_ts),
        "order_approved_at": None,
        "order_delivered_carrier_date": None,
        "order_delivered_customer_date": None,
        "order_estimated_delivery_date": fmt(purchase_ts + timedelta(days=random.randint(7, 30))),
    })

    total = Decimal("0")
    for item_number in range(1, random.randint(1, 3) + 1):
        item = random.choice(items_pool)
        new_items.append({
            "order_id": order_id,
            "order_item_id": str(item_number),
            "product_id": item.product_id,
            "seller_id": item.seller_id,
            "shipping_limit_date": fmt(purchase_ts + timedelta(days=3)),
            "price": str(item.price),
            "freight_value": str(item.freight_value),
        })
        total += item.price + item.freight_value

    payment_type = random.choice(["credit_card", "boleto", "voucher", "debit_card"])
    new_payments.append({
        "order_id": order_id,
        "payment_sequential": "1",
        "payment_type": payment_type,
        "payment_installments": str(random.randint(1, 10) if payment_type == "credit_card" else 1),
        "payment_value": str(total),
    })

print(f"{len(new_orders)} orders, {len(new_items)} items, {len(new_payments)} payments generated")

In [0]:
NEXT_STATUS = {
    "created": "approved",
    "approved": "invoiced",
    "invoiced": "processing",
    "processing": "shipped",
    "shipped": "delivered",
}

candidates = (
    spark.table(f"{SILVER}.orders")
    .filter(F.col("order_status").isin(list(NEXT_STATUS.keys())))
    .orderBy(F.rand())
    .limit(N_STATUS_UPDATES)
    .collect()
)

updated_orders = []
for order in candidates:
    row = {
        col: fmt(value) if isinstance(value, datetime) else value
        for col, value in order.asDict().items()
        if col in COLUMNS["orders"]
    }
    new_status = NEXT_STATUS[row["order_status"]]
    row["order_status"] = new_status

    if new_status == "approved":
        row["order_approved_at"] = fmt(NOW)
    elif new_status == "shipped":
        row["order_delivered_carrier_date"] = fmt(NOW)
    elif new_status == "delivered":
        row["order_delivered_customer_date"] = fmt(NOW)

    updated_orders.append(row)

print(f"{len(updated_orders)} orders got their status updated")

In [0]:
bad_orders = []
# generating bad records on purpose to check if silve layer DQ works
if INJECT_BAD_RECORDS:
    bad_orders = [
        {   
            "order_id": None,
            "customer_id": uuid.uuid4().hex,
            "order_status": "created",
            "order_purchase_timestamp": fmt(NOW),
            "order_approved_at": None,
            "order_delivered_carrier_date": None,
            "order_delivered_customer_date": None,
            "order_estimated_delivery_date": fmt(NOW + timedelta(days=10)),
        },
        {   
            "order_id": uuid.uuid4().hex,
            "customer_id": uuid.uuid4().hex,
            "order_status": "created",
            "order_purchase_timestamp": "31/02/2026 25:61:00",
            "order_approved_at": None,
            "order_delivered_carrier_date": None,
            "order_delivered_customer_date": None,
            "order_estimated_delivery_date": fmt(NOW + timedelta(days=10)),
        },
    ]

print(f"{len(bad_orders)} bad records added")

In [0]:
def write_batch(records: list[dict], entity: str) -> int:
    if not records:
        return 0

    columns = COLUMNS[entity]
    schema = StructType([StructField(c, StringType(), True) for c in columns])
    rows = [tuple(record.get(c) for c in columns) for record in records]

    (
        spark.createDataFrame(rows, schema)
        .coalesce(1)                          
        .write
        .option("ignoreNullFields", "false") 
        .mode("append")
        .json(f"{INCREMENTAL_PATH}/{entity}/batch_{BATCH_ID}")
    )
    return len(records)

print("orders:        ", write_batch(new_orders + updated_orders + bad_orders, "orders"))
print("order_items:   ", write_batch(new_items, "order_items"))
print("order_payments:", write_batch(new_payments, "order_payments"))
print("customers:     ", write_batch(new_customers, "customers"))
print(f"Batch {BATCH_ID} saved in {INCREMENTAL_PATH}")

In [0]:
%sql
LIST '/Volumes/olist_lakehouse/landing/olist/incremental/orders/'